In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics  import  (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
phones_df = pd.read_csv('datasets/transformed_data/phones.csv')

X_data = phones_df.drop(columns=['price'])
y_data = phones_df['price']

X, X_test, y, y_test = train_test_split(
    X_data,
    y_data,
    test_size=.1
)

In [3]:
GEN_SIZE = 1
POP_SIZE = 15
MUTATION_PROB = 0.2
CROSSOVER_PROB = 0.8
SIGMA = 0.1
N_FEATURES = len(X_data.columns)

In [4]:
population = []

for _ in range(POP_SIZE):
    feature_mask = np.random.randint(
        0,
        2,
        N_FEATURES
    )
    
    if feature_mask.sum() == 0:
        feature_mask[np.random.randint(N_FEATURES)] = 1
    
    chromosome = np.concatenate([
        feature_mask,
        [
            np.random.randint(50, 501),
            np.random.randint(3, 31),
            np.random.randint(2, 16),
            np.random.randint(1, 11),
            np.random.randint(0, 3)
        ]]
    )
    
    population.append(chromosome)
    
population = np.array(population,dtype=object)

In [5]:

def decode(chromosome):
    
    feature_mask = chromosome[:N_FEATURES].astype(bool)
    
    n_estimators = int(chromosome[N_FEATURES])    
    max_depth = int(chromosome[N_FEATURES + 1])
    min_split = int(chromosome[N_FEATURES + 2])
    min_leaf = int(chromosome[N_FEATURES + 3])
    max_features_gene = int(chromosome[N_FEATURES + 4])
    
    max_features_map = {
        0:"sqrt",
        1:"log2",
        2: None
    }
    
    params = {
        "n_estimators" : n_estimators,
        "max_depth" : max_depth,
        "min_samples_split": min_split,
        "min_samples_leaf": min_leaf,
        "max_features": max_features_map[max_features_gene]
    }
    
    
    return feature_mask, params 

In [6]:

mask, params = decode(population[0])

mask, params

(array([False,  True,  True, False,  True, False, False, False, False,
         True, False, False, False, False, False,  True, False, False,
        False,  True,  True,  True, False,  True, False, False,  True,
        False, False, False,  True,  True, False,  True,  True,  True,
         True, False,  True,  True,  True, False, False, False,  True,
         True,  True,  True, False,  True,  True]),
 {'n_estimators': 264,
  'max_depth': 15,
  'min_samples_split': 12,
  'min_samples_leaf': 8,
  'max_features': 'log2'})

In [7]:

from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from cuml.ensemble import RandomForestRegressor
global_best_score = 0
global_best_chromosome = []

kf = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [8]:

def fitness_function(chromosome, X, y):
    feature_mask, params = decode(chromosome)
    
    if(feature_mask.sum()) == 0:
        return
    
    X_selected = X.loc[:, feature_mask]
    
    model = RandomForestRegressor(
        **params,
        random_state=42
    )

    scores = cross_val_score(
        model,
        X_selected,
        y,
        cv=kf,
        scoring="r2"
    )
    print(scores)
    return np.mean(scores)

In [9]:
best_history = []
mean_history = []

In [10]:

def tournament_selection(population, fitness, k=3):
    selected = []
    
    pop_size = len(population)
    
    for _ in range(pop_size):
        
        idx = np.random.choice(
            pop_size,
            size=k,
            replace=False
        )
        
        tournament_fitness = fitness[idx]
        
        winner = idx[np.argmax(tournament_fitness)]
        
        selected.append(population[winner].copy())
        
    return np.array(selected, dtype=object)


def roulette_selection(population, fitness):
    
    adjusted = fitness - fitness.min() + 1e-10
    
    probability = adjusted / adjusted.sum()
    
    selected_index = np.random.choice(
        len(population),
        size = len(population),
        replace=True,
        p=probability
    )
    
    return np.array(
        [population[i].copy() for i in selected_index],
        dtype=object
    )
    
    
def boltzmann_selection(population, fitness, temperature):
    
    scaled = np.exp(
        (fitness-np.max(fitness)) / temperature
    )
    
    probability = scaled / scaled.sum()
    
    selected_index = np.random.choice(
        len(population),
        size=len(population),
        replace=True,
        p=probability
    )
    
    return np.array(
        [population[i].copy() for i in selected_index],
        dtype=object
    )

In [11]:
def crossover(parent1, parent2):
    child1 = parent1.copy()
    child2 = parent2.copy()
    
    for i in range(N_FEATURES):
        if np.random.rand() < 0.5:
            child1[i], child2[i] = child2[i], child1[i]
            
    for i in range(N_FEATURES, len(parent1)):
        if np.random.rand() < 0.5:
            child1[i], child2[i] = child2[i], child1[i]
            
    return child1, child2

In [12]:
def gaussian_mutation(value,  lower, upper):
    std = SIGMA * (upper - lower)
    value += np.random.normal(0, std)
    value = np.clip(value, lower, upper)
    
    return int(round(value))


def mutate(chromosome):
    chromosome = chromosome.copy()
    
    for i in range(N_FEATURES):
        if np.random.rand() < MUTATION_PROB:
            chromosome[i] = 1 - chromosome[i]
            
    if np.random.rand() < MUTATION_PROB:
        chromosome[N_FEATURES ] = gaussian_mutation(
            chromosome[N_FEATURES ], 50, 500
        )
    if np.random.rand() < MUTATION_PROB:
        chromosome[N_FEATURES + 1] = gaussian_mutation(
            chromosome[N_FEATURES + 1], 2, 50
        )
    if np.random.rand() < MUTATION_PROB:
        chromosome[N_FEATURES + 2] = gaussian_mutation(
            chromosome[N_FEATURES + 2], 2, 20
        )
    if np.random.rand() < MUTATION_PROB:
        chromosome[N_FEATURES + 3] = gaussian_mutation(
            chromosome[N_FEATURES + 3], 1, 20
        )
    if np.random.rand() < MUTATION_PROB:
        chromosome[N_FEATURES + 4] = np.random.randint(3)
        
    return chromosome

In [13]:
import time 

start = time.perf_counter()

for generation in range(GEN_SIZE):
    print(f"Generation {generation}")
    fitness = np.array([
        fitness_function(chromosome, X, y )
        for chromosome in population
    ])
    best_history.append(fitness.max())
    mean_history.append(fitness.mean())

    best_idx = np.argmax(fitness)
    if fitness[best_idx] > global_best_score:
        global_best_score = fitness[best_idx]
        global_best = population[best_idx].copy()

    parents = tournament_selection(population, fitness, k=3)
    print(f"Fitness: {fitness.max():.4f}")
    offspring = []
    for j in range(0, POP_SIZE - 1, 2):

        parent1 = parents[j]
        parent2 = parents[j+1]
        
        if np.random.rand() < CROSSOVER_PROB:
            child1, child2 = crossover(parent1, parent2)
        else:
            child1 = parent1.copy()
            child2 = parent2.copy()
        
        child1 = mutate(child1)
        child2 = mutate(child2)
        
        offspring.extend([child1, child2])
    # offspring.append([child1, child2])
    population = np.array(offspring, dtype=object)

end = time.perf_counter()

Generation 0
[0.73901099 0.78625499 0.81290236 0.70118884 0.73774797 0.7156447
 0.79263428 0.73161561 0.67002537 0.14206043]
[0.6849308  0.79510236 0.78861482 0.70937605 0.73983802 0.71777551
 0.7572646  0.68638539 0.67342695 0.13801001]
[0.72456545 0.80223076 0.82394303 0.7306674  0.76703465 0.68794543
 0.79675584 0.75651144 0.6676796  0.14281492]
[0.73729608 0.80394915 0.81252622 0.73826416 0.72459039 0.70063429
 0.78272806 0.69380545 0.68333873 0.13884489]
[0.74760896 0.81261526 0.81672639 0.69728218 0.74500371 0.71689602
 0.791587   0.72824805 0.66937037 0.14316252]
[0.7309379  0.81418928 0.81516559 0.69690385 0.73942232 0.71482839
 0.78918016 0.72496826 0.67219338 0.14207741]
[0.73995619 0.80150267 0.82083112 0.71071412 0.75242232 0.68990324
 0.7982262  0.7434261  0.67676694 0.14208563]
[0.69754487 0.78068264 0.78519324 0.70982912 0.71360776 0.7109032
 0.74308816 0.67912416 0.67155707 0.13637371]
[0.73974674 0.80916027 0.81674001 0.69903003 0.73772943 0.71515978
 0.79681915 0.7340

In [14]:
print(f"Waktu eksekusi pelatihan model: {(end - start):.2f}")

Waktu eksekusi pelatihan model: 343.05
